## Exploratory notebook setup

See `docs/notebooks.md` for inputs, execution order, and environment setup.
Run a working copy prepared by `tools/prepare_workspace.py`.
Saved outputs were removed from this copy; executed originals remain on the NAS.
The following cell changes file locations only. The analysis formulas are preserved.


In [ ]:
import os
from pathlib import Path

# Set these before launching Jupyter when using another storage mount.
VQ_NAS_ROOT = Path(os.environ.get("VQ_NAS_ROOT", "/megaverse/storage/nasiri"))
VQ_JUPYTER_ROOT = Path(os.environ.get("VQ_JUPYTER_ROOT", str(VQ_NAS_ROOT / "jupyter")))


# Encoder Locality Analysis

## Goal (H1: patch pixel-noise on encoder)
We perturb the **input image** with Gaussian noise **only inside a pixel-space patch** and re-encode to discrete token IDs.
The question is: **how often do token IDs change outside the token-aligned patch region?**

Each sample provides:
- `token_grid_hw = (H, W)` token grid resolution
- `patch_bbox_px = (x0, y0, x1, y1)` pixel-space patch (where noise was applied)
- `patch_bbox_tok = (j0, i0, j1, i1)` token-space bounding box aligned to the token grid
- `indices_clean`, `indices_low`, `indices_mid`, `indices_high`: flattened token IDs (length `H*W`)


## Setup

In [ ]:
from pathlib import Path
import glob
import json
import pandas as pd
import numpy as np
import random
from PIL import Image
import matplotlib.pyplot as plt

OUTDIR = VQ_JUPYTER_ROOT / "1765986549_robustness_dataset_vqgan_h1_patch_noise_encoder"

meta_files = sorted(OUTDIR.glob("metadata_part_*.jsonl"))
print("OUTDIR:", OUTDIR)
print("Found metadata files:", len(meta_files))
for p in meta_files[:10]:
    print(" -", p.name)
assert len(meta_files) > 0, "No metadata_part_*.jsonl files found. Fix OUTDIR."


In [ ]:
MAX_RECORDS = None  # set e.g. 2000 for a quick first pass

records = []
for fp in meta_files:
    with open(fp, "r") as f:
        for line in f:
            records.append(json.loads(line))
            if MAX_RECORDS is not None and len(records) >= MAX_RECORDS:
                break
    if MAX_RECORDS is not None and len(records) >= MAX_RECORDS:
        break

df = pd.DataFrame(records)
print("Loaded records:", len(df))
print("Columns:", list(df.columns))
df.head(3)


In [ ]:
IMAGES_DIR = OUTDIR / "images"

img_files = sorted([p for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in {".png"}])
print("IMAGES_DIR:", IMAGES_DIR)
print("Total image files found:", len(img_files))
print("Sample filenames:")
for p in random.sample(img_files, k=min(15, len(img_files))):
    print(" -", p.relative_to(IMAGES_DIR))


### Sanity Check

In [ ]:
required = ["indices_clean", "indices_low", "indices_mid", "indices_high",
            "patch_bbox_tok", "token_grid_hw", "noise_std"]
missing = [k for k in required if k not in df.columns]
assert not missing, f"Missing required fields: {missing}"

# Normalise noise_std to a tuple (hashable) and also keep explicit named stds
df["noise_std_tuple"] = df["noise_std"].apply(tuple)

# Most runs have a single triple like (low, mid, high); we assert that here
uniq_noise = df["noise_std_tuple"].unique()
print("Unique noise_std tuples:", uniq_noise[:10])
assert len(uniq_noise) == 1, f"Expected one noise_std triple; got {len(uniq_noise)}"

sigma_low, sigma_mid, sigma_high = uniq_noise[0]
print("σ_low, σ_mid, σ_high =", sigma_low, sigma_mid, sigma_high)

print("token_grid_hw unique (first 10):", df["token_grid_hw"].astype(str).unique()[:10])

# Check a couple rows manually
for i in range(min(3, len(df))):
    H, W = df.loc[i, "token_grid_hw"]
    bbox = df.loc[i, "patch_bbox_tok"]
    print(f"row {i}: grid=({H},{W})  bbox={bbox}  len(clean)={len(df.loc[i,'indices_clean'])}")


In [ ]:
def _bbox_area(b):
    j0, i0, j1, i1 = b
    return max(0, j1 - j0) * max(0, i1 - i0)

H = df["token_grid_hw"].apply(lambda x: x[0]).to_numpy()
W = df["token_grid_hw"].apply(lambda x: x[1]).to_numpy()
HW = H * W

len_clean = df["indices_clean"].apply(len).to_numpy()
len_low   = df["indices_low"].apply(len).to_numpy()
len_mid   = df["indices_mid"].apply(len).to_numpy()
len_high  = df["indices_high"].apply(len).to_numpy()

assert np.all(len_clean == HW), "Some indices_clean lengths != H*W"
assert np.all(len_low == HW) and np.all(len_mid == HW) and np.all(len_high == HW), "Some noisy indices lengths != H*W"

bboxes = df["patch_bbox_tok"].to_list()
j0 = np.array([b[0] for b in bboxes])
i0 = np.array([b[1] for b in bboxes])
j1 = np.array([b[2] for b in bboxes])
i1 = np.array([b[3] for b in bboxes])

assert np.all((0 <= j0) & (j0 <= j1) & (j1 <= W)), "BBox x-bounds invalid for some rows"
assert np.all((0 <= i0) & (i0 <= i1) & (i1 <= H)), "BBox y-bounds invalid for some rows"

patch_area = np.array([_bbox_area(b) for b in bboxes])
patch_frac = patch_area / HW

print("Patch fraction token-space summary:")
print(pd.Series(patch_frac).describe(percentiles=[0.1,0.5,0.9,0.99]))

## Visual Sanity Check

In [ ]:
def find_variants(image_id: str):
    """
    Try to find 1-4 images in OUTDIR/images that belong to this image_id.
    Returns dict with keys among: clean, low, mid, high, any
    """
    # Broad search: any file containing the image_id
    hits = [p for p in img_files if image_id in str(p)]
    out = {}

    if not hits:
        return out

    # Heuristic bucketing by filename
    for p in hits:
        s = p.name.lower()
        if "clean" in s:
            out["clean"] = p
        elif "low" in s:
            out["low"] = p
        elif "mid" in s:
            out["mid"] = p
        elif "high" in s:
            out["high"] = p

    # Fallback: if none matched buckets, take first as "any"
    if not out:
        out["any"] = hits[0]

    return out

def draw_boxes(ax, img_w, img_h, bbox_px, bbox_tok, grid_hw):
    # bbox_px: [x0, y0, x1, y1] in pixels
    x0, y0, x1, y1 = bbox_px
    ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0,
                              fill=False, linewidth=2))

    # token bbox projected to pixel coords (cyan dashed)
    Ht, Wt = grid_hw
    j0, i0, j1, i1 = bbox_tok
    sx = img_w / Wt
    sy = img_h / Ht
    tx0, ty0 = j0 * sx, i0 * sy
    tx1, ty1 = j1 * sx, i1 * sy
    ax.add_patch(plt.Rectangle((tx0, ty0), tx1-tx0, ty1-ty0,
                              fill=False, linewidth=2, linestyle="--"))

def show_variant(ax, path, title, row):
    img = Image.open(path).convert("RGB")
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    draw_boxes(ax, img.width, img.height,
               row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])


In [ ]:
def find_full_set(image_id: str):
    # Return dict idx(int)->Path for files "0_*.png" ... "7_*.png" under that image_id folder
    # We rely on the exporter’s numbering convention.
    folder_hits = [p for p in img_files if (image_id in str(p))]
    if not folder_hits:
        return {}

    # usually all 8 sit in one folder; infer folder from any hit
    parent = folder_hits[0].parent
    out = {}
    for k in range(8):
        matches = sorted(parent.glob(f"{k}_*.png"))
        if matches:
            out[k] = matches[0]
    return out


In [ ]:
def show_sample_grid(row, overlay=True):
    image_id = row["image_id"]
    files = find_full_set(image_id)

    print("\nimage_id:", image_id)
    print("found:", {k: files[k].name for k in sorted(files.keys())})

    needed = list(range(8))
    if not all(k in files for k in needed):
        print("WARNING: not all 0..7 found for this sample.")
        return

    titles_top = ["original", "input low", "input mid", "input high"]
    titles_bot = ["recon clean", "recon low", "recon mid", "recon high"]

    top_paths = [files[0], files[2], files[4], files[6]]
    bot_paths = [files[1], files[3], files[5], files[7]]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for c, p in enumerate(top_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[0, c]
        ax.imshow(img)
        ax.set_title(titles_top[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_boxes(ax, img.width, img.height,
                       row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])

    for c, p in enumerate(bot_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[1, c]
        ax.imshow(img)
        ax.set_title(titles_bot[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_boxes(ax, img.width, img.height,
                       row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])

    plt.tight_layout()
    plt.show()

N = 3
for r in df.sample(N, random_state=2).to_dict(orient="records"):
    show_sample_grid(r, overlay=True)


## Definitions and Metrics

Let the clean token grid be  
$ \mathbf{z} \in \{0,\dots,K-1\}^{H \times W} $  
and the noisy token grid at noise level $\sigma$ be  
$ \mathbf{z}_{\sigma} $.

Define the per-token change indicator:
$$
\Delta_{\sigma}(i,j)
= \mathbf{1}\!\left[ \mathbf{z}_{\sigma}(i,j) \neq \mathbf{z}(i,j) \right].
$$

Let the token patch box be  
$ B = (j_0,i_0,j_1,i_1) $  
and define the set of token positions inside it:
$$
\mathcal{P}
= \{(i,j)\mid i_0 \le i < i_1,\; j_0 \le j < j_1\},
\qquad
\mathcal{P}^c \text{ its complement.}
$$

**Inside-change**
$$
\text{inside\_change}(\sigma)
= \frac{1}{|\mathcal{P}|}
\sum_{(i,j)\in\mathcal{P}} \Delta_{\sigma}(i,j)
$$

**Outside-change (leakage / non-locality)**
$$
\text{outside\_change}(\sigma)
= \frac{1}{|\mathcal{P}^c|}
\sum_{(i,j)\in\mathcal{P}^c} \Delta_{\sigma}(i,j)
$$

**Leakage ratio**
$$
\text{leakage\_ratio}(\sigma)
=
\frac{\text{outside\_change}(\sigma)}
{\text{inside\_change}(\sigma) + \varepsilon}
$$

with a small $\varepsilon > 0$ to avoid division by zero.

In [ ]:
# Grid is constant in your run (16x16), but we keep it general.
H = df["token_grid_hw"].apply(lambda x: x[0]).to_numpy()
W = df["token_grid_hw"].apply(lambda x: x[1]).to_numpy()
HW = H * W

assert np.all(HW == HW[0]),
H0, W0, HW0 = int(H[0]), int(W[0]), int(HW[0])

# Build inside mask per sample: shape (N, HW)
N = len(df)
inside_mask = np.zeros((N, HW0), dtype=bool)

bboxes = df["patch_bbox_tok"].to_list()
for n, (j0, i0, j1, i1) in enumerate(bboxes):
    # Token grid is (H0, W0): row=i, col=j
    m2d = inside_mask[n].reshape(H0, W0)
    m2d[i0:i1, j0:j1] = True

inside_count = inside_mask.sum(axis=1).astype(np.float32)
outside_count = (HW0 - inside_count).astype(np.float32)

print("Token grid:", (H0, W0), "HW=", HW0)
print("Inside token count summary:", np.percentile(inside_count, [0, 10, 50, 90, 99, 100]))
print("Outside token count summary:", np.percentile(outside_count, [0, 10, 50, 90, 99, 100]))


In [ ]:
def compute_change_metrics(noisy_col: str, eps: float = 1e-8):
    z_clean = np.asarray(df["indices_clean"].to_list(), dtype=np.int32)   # (N, HW)
    z_noisy = np.asarray(df[noisy_col].to_list(), dtype=np.int32)         # (N, HW)

    delta = (z_noisy != z_clean)  # bool (N, HW)

    inside_changed = (delta & inside_mask).sum(axis=1).astype(np.float32)
    outside_changed = (delta & (~inside_mask)).sum(axis=1).astype(np.float32)

    inside_change = inside_changed / inside_count
    outside_change = outside_changed / outside_count
    leakage_ratio = outside_change / (inside_change + eps)

    return inside_change, outside_change, leakage_ratio

inside_low, outside_low, ratio_low   = compute_change_metrics("indices_low")
inside_mid, outside_mid, ratio_mid   = compute_change_metrics("indices_mid")
inside_high, outside_high, ratio_high = compute_change_metrics("indices_high")

print("Computed change metrics for low/mid/high.")


In [ ]:
import pandas as pd
import numpy as np

def summarize(arr):
    return {
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "q90": float(np.quantile(arr, 0.90)),
        "q99": float(np.quantile(arr, 0.99)),
    }

summary = pd.DataFrame({
    "sigma": [sigma_low, sigma_mid, sigma_high],
    "inside_change": [summarize(inside_low), summarize(inside_mid), summarize(inside_high)],
    "outside_change": [summarize(outside_low), summarize(outside_mid), summarize(outside_high)],
    "leakage_ratio": [summarize(ratio_low), summarize(ratio_mid), summarize(ratio_high)],
})

# Expand dict columns into columns
def expand(col):
    return pd.DataFrame(summary[col].tolist()).add_prefix(f"{col}_")

out = pd.concat([summary[["noise sigma"]], expand("inside_change"), expand("outside_change"), expand("leakage_ratio")], axis=1)
out


**Results: Locality vs Leakage**:

We quantify encoder locality by measuring the fraction of token IDs that change under patch-localised pixel noise.

**Inside response**:
Inside the token-aligned patch, token changes are already high at low noise and quickly saturate:
- At $\sigma=0.1$, the median inside-change is $\approx 0.77$.
- At $\sigma=0.5$, the median inside-change is $\approx 0.99$ (near-complete change).

This indicates that patch noise strongly perturbs local token assignments, and by $\sigma=0.5$ the patch region is effectively saturated (little headroom remains).

**Outside response**:
Outside the patch, token changes are substantial and increase with noise:
- At $\sigma=0.1$, the median outside-change is $\approx 0.27$.
- At $\sigma=0.5$, the median outside-change is $\approx 0.49$.

Thus, a perturbation confined to a local pixel patch induces widespread token changes beyond the patch region, suggesting non-local encoder sensitivity.

**Normalised leakage ratio**:

The median leakage ratio increases with noise:
- $\sigma=0.1$: $\approx 0.36$
- $\sigma=0.2$: $\approx 0.42$
- $\sigma=0.5$: $\approx 0.50$

At high noise, the outside region exhibits roughly half as many token changes as the inside region, which is inconsistent with strictly local behaviour and indicates significant non-local propagation effects.

**Tail behaviour**:
The 99th percentile outside-change reaches $\approx 0.70$ at $\sigma=0.5$, showing that a small subset of images experiences extremely strong leakage.


## Spatial Leakage Profile: change probability vs distance from patch

Scalar outside-change cannot distinguish *boundary spillover* from *long-range propagation*.
We therefore compute a distance-to-patch curve.

For each token position $(i,j)$ on the $H \times W$ grid, define its Manhattan distance to the patch:
- $d(i,j)=0$ if $(i,j)\in\mathcal{P}$
- otherwise $d(i,j)$ is the Manhattan distance to the nearest patch boundary.

We then estimate, for each noise level $\sigma$, the conditional change probability
$$
p_{\sigma}(d)
=
\mathbb{E}\left[
\frac{1}{|\{(i,j): d(i,j)=d\}|}
\sum_{(i,j): d(i,j)=d}
\Delta_{\sigma}(i,j)
\right],
\qquad d=0,1,2,\dots
$$

If leakage is mostly local, $p_{\sigma}(d)$ should drop sharply after $d=0$ (and ideally after $d=1$).
A slow decay indicates long-range non-locality.


In [ ]:
import numpy as np

# Precompute a base (i,j) grid once (since H,W are constant here)
ii, jj = np.meshgrid(np.arange(H0), np.arange(W0), indexing="ij")  # (H,W)

bboxes = df["patch_bbox_tok"].to_list()
dist_map = np.zeros((N, HW0), dtype=np.int16)

for n, (j0, i0, j1, i1) in enumerate(bboxes):
    # distance to interval [i0, i1-1] in rows, [j0, j1-1] in cols
    # (0 if inside)
    di = np.where(ii < i0, i0 - ii, np.where(ii > (i1 - 1), ii - (i1 - 1), 0))
    dj = np.where(jj < j0, j0 - jj, np.where(jj > (j1 - 1), jj - (j1 - 1), 0))
    d = (di + dj).astype(np.int16)  # Manhattan distance to the box
    dist_map[n] = d.reshape(-1)

max_d = int(dist_map.max())
print("Max Manhattan distance observed:", max_d)
print("Distances present:", np.unique(dist_map)[:20], "...", np.unique(dist_map)[-10:])


In [ ]:
def p_of_d(noisy_col: str):
    z_clean = np.asarray(df["indices_clean"].to_list(), dtype=np.int32)
    z_noisy = np.asarray(df[noisy_col].to_list(), dtype=np.int32)
    delta = (z_noisy != z_clean)  # (N, HW)

    p = []
    counts = []
    for d in range(max_d + 1):
        mask = (dist_map == d)  # (N, HW)
        cnt = mask.sum()
        counts.append(int(cnt))
        if cnt == 0:
            p.append(np.nan)
        else:
            p.append(float(delta[mask].mean()))
    return np.array(p, dtype=np.float32), np.array(counts, dtype=np.int64)

p_low, cnt_d = p_of_d("indices_low")
p_mid, _     = p_of_d("indices_mid")
p_high, _    = p_of_d("indices_high")

# Show a compact table for first few distances
d_show = max_d + 1
pd.DataFrame({
    "d": np.arange(d_show),
    "count_tokens_total": cnt_d[:d_show],
    f"p(d)|σ={sigma_low}": p_low[:d_show],
    f"p(d)|σ={sigma_mid}": p_mid[:d_show],
    f"p(d)|σ={sigma_high}": p_high[:d_show],
})


In [ ]:
d = np.arange(max_d+1)

plt.figure(figsize=(7, 4))
plt.plot(d, p_low,  label=f"σ={sigma_low}")
plt.plot(d, p_mid,  label=f"σ={sigma_mid}")
plt.plot(d, p_high, label=f"σ={sigma_high}")
plt.xlabel("Manhattan distance to patch (tokens)")
plt.ylabel("P(token ID changes)")
plt.title("Spatial leakage profile")
plt.legend()
plt.ylim(0, 1)
plt.yticks(np.arange(0.0, 1.01, 0.1))
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


**Results: Spatial leakage profile**

We compute the distance-conditioned change probability $p_{\sigma}(d)$, where $d$ is Manhattan distance (in token units) to the token-aligned patch box.

Across all noise levels, $p_{\sigma}(d)$ decreases sharply from $d=0$ to small distances ($d\approx 1\text{–}3$), indicating that the patch region is the most affected as expected.

However, $p_{\sigma}(d)$ does **not** decay toward zero for larger distances. Instead, it approaches a non-zero plateau:
- for $\sigma=0.1$, $p_{\sigma}(d)$ stabilizes around $\approx 0.16$ at large $d$,
- for $\sigma=0.2$, the plateau is around $\approx 0.22$,
- for $\sigma=0.5$, the plateau is around $\approx 0.31$.

This behaviour indicates substantial **non-local propagation**: a perturbation confined to a local pixel patch causes widespread token ID changes far from the patch, and the magnitude of this global effect increases with noise strength $\sigma$.

Note that extreme distances (largest $d$) involve very few token positions, so tail fluctuations are not statistically reliable.


### Boundary vs Far-field Leakage Decomposition

The spatial profile suggests two regimes:
1) a sharp drop close to the patch (boundary spillover),
2) a non-zero plateau at larger distances (far-field/global effect).

To quantify this, we decompose outside changes by distance rings:
- ring-$d$: tokens with Manhattan distance exactly $d$
- near-field: $1 \le d \le d_\text{near}$
- far-field: $d \ge d_\text{far}$

This separates “just the boundary” from truly long-range leakage.


In [ ]:
# choose bands (reasonable defaults for 16x16)
D_NEAR = 2   # near-field = d in {1,2}
D_FAR  = 6   # far-field  = d >= 6

def band_mean(p, dmin=None, dmax=None):
    d = np.arange(len(p))
    mask = np.ones_like(d, dtype=bool)
    if dmin is not None:
        mask &= (d >= dmin)
    if dmax is not None:
        mask &= (d <= dmax)
    # ignore NaNs (shouldn't happen except maybe extremes)
    return float(np.nanmean(p[mask]))

summary_bands = pd.DataFrame([
    {
        "sigma": sigma_low,
        "p(d=0) inside": float(p_low[0]),
        "p(d=1) boundary": float(p_low[1]),
        "p(d=2)": float(p_low[2]),
        f"near mean (d=1..{D_NEAR})": band_mean(p_low, 1, D_NEAR),
        f"far mean (d>={D_FAR})": band_mean(p_low, D_FAR, max_d),
        f"plateau mean (d=8..12)": band_mean(p_low, 8, 12),
    },
    {
        "sigma": sigma_mid,
        "p(d=0) inside": float(p_mid[0]),
        "p(d=1) boundary": float(p_mid[1]),
        "p(d=2)": float(p_mid[2]),
        f"near mean (d=1..{D_NEAR})": band_mean(p_mid, 1, D_NEAR),
        f"far mean (d>={D_FAR})": band_mean(p_mid, D_FAR, max_d),
        f"plateau mean (d=8..12)": band_mean(p_mid, 8, 12),
    },
    {
        "sigma": sigma_high,
        "p(d=0) inside": float(p_high[0]),
        "p(d=1) boundary": float(p_high[1]),
        "p(d=2)": float(p_high[2]),
        f"near mean (d=1..{D_NEAR})": band_mean(p_high, 1, D_NEAR),
        f"far mean (d>={D_FAR})": band_mean(p_high, D_FAR, max_d),
        f"plateau mean (d=8..12)": band_mean(p_high, 8, 12),
    },
])

summary_bands


In [ ]:
def per_image_band_changes(noisy_col: str, d_near=2, d_far=6):
    z_clean = np.asarray(df["indices_clean"].to_list(), dtype=np.int32)
    z_noisy = np.asarray(df[noisy_col].to_list(), dtype=np.int32)
    delta = (z_noisy != z_clean)  # (N, HW)

    d = dist_map  # (N, HW)

    # masks
    inside = (d == 0)
    boundary = (d == 1)
    near = (d >= 1) & (d <= d_near)
    far = (d >= d_far)

    def frac(mask):
        cnt = mask.sum(axis=1).astype(np.float32)
        changed = (delta & mask).sum(axis=1).astype(np.float32)
        return changed / cnt

    return {
        "inside": frac(inside),
        "boundary": frac(boundary),
        f"near_1_{d_near}": frac(near),
        f"far_ge_{d_far}": frac(far),
    }

bands_low  = per_image_band_changes("indices_low",  d_near=D_NEAR, d_far=D_FAR)
bands_mid  = per_image_band_changes("indices_mid",  d_near=D_NEAR, d_far=D_FAR)
bands_high = per_image_band_changes("indices_high", d_near=D_NEAR, d_far=D_FAR)

def summarize_band_dict(bdict):
    return {k: {
        "mean": float(np.mean(v)),
        "median": float(np.median(v)),
        "q90": float(np.quantile(v, 0.90)),
        "q99": float(np.quantile(v, 0.99)),
    } for k, v in bdict.items()}

band_summary = pd.DataFrame({
    "sigma": [sigma_low, sigma_mid, sigma_high],
    "bands": [summarize_band_dict(bands_low), summarize_band_dict(bands_mid), summarize_band_dict(bands_high)]
})

# Expand to a readable table
rows = []
for _, r in band_summary.iterrows():
    s = r["sigma"]
    for band, stats in r["bands"].items():
        rows.append({
            "sigma": s,
            "band": band,
            **{f"{k}": stats[k] for k in stats}
        })

pd.DataFrame(rows).sort_values(["sigma", "band"])


In [ ]:
# earlier outside_change arrays: outside_low/mid/high
# compute outside from dist_map directly (d>=1)
def outside_from_dist(noisy_col: str):
    z_clean = np.asarray(df["indices_clean"].to_list(), dtype=np.int32)
    z_noisy = np.asarray(df[noisy_col].to_list(), dtype=np.int32)
    delta = (z_noisy != z_clean)
    outside = (dist_map >= 1)
    return (delta & outside).sum(axis=1) / outside.sum(axis=1)

o_low2  = outside_from_dist("indices_low")
o_mid2  = outside_from_dist("indices_mid")
o_high2 = outside_from_dist("indices_high")

print("Mean outside_change (previous) vs recomputed via dist_map:")
print("low :", float(np.mean(outside_low)),  float(np.mean(o_low2)))
print("mid :", float(np.mean(outside_mid)),  float(np.mean(o_mid2)))
print("high:", float(np.mean(outside_high)), float(np.mean(o_high2)))


**Results: Boundary vs Far-field Leakage**

To distinguish boundary spillover from long-range effects, we decompose token changes by distance from the patch using the Manhattan distance $d(i,j)$ in token units.

We report per-image change fractions in different bands:
- **Inside**: $d=0$
- **Boundary ring**: $d=1$
- **Near-field**: $1 \le d \le 2$
- **Far-field**: $d \ge 6$

**Near-field (boundary spillover)**:

Near the patch, change rates remain high:
- at $\sigma=0.1$, the mean near-field change ($1 \le d \le 2$) is $\approx 0.424$,
- at $\sigma=0.5$, it increases to $\approx 0.728$.

This indicates strong spillover around the patch boundary.

**Far-field (global effect)**

Crucially, changes persist far away from the patch:
- at $\sigma=0.1$, far-field change ($d \ge 6$) has mean $\approx 0.164$ (median $\approx 0.154$),
- at $\sigma=0.2$, mean $\approx 0.233$,
- at $\sigma=0.5$, mean $\approx 0.326$ (median $\approx 0.314$).

Thus, even well-separated tokens exhibit non-trivial change probability, which increases with $\sigma$.
This behaviour is inconsistent with purely local influence and indicates a substantial global/non-local sensitivity of the encoder token assignments.

We additionally observe a plateau in $p_{\sigma}(d)$ for large distances (e.g., $d \in [8,12]$), whose mean rises from $\approx 0.157$ at $\sigma=0.1$ to $\approx 0.317$ at $\sigma=0.5$.


## Qualitative Visual Examples (Token ID change mask)

This visualization shows **where discrete token IDs changed** after injecting noise, compared to the clean encoding.

Each cell corresponds to one token position in the (H \times W) latent grid:

* **Yellow (1)**: the token ID at this position changed due to noise.
* **Purple (0)**: the token ID remained unchanged.

The black rectangle indicates the **edited patch region** in token space.
Changes inside the box reflect the **direct effect** of the injected noise, while changes outside the box reveal **non-local effects (spatial leakage)** of the encoder–decoder pipeline.

In [ ]:
import numpy as np

rank_ratio = ratio_mid  # or ratio_low / ratio_high

idx_sorted = np.argsort(rank_ratio)
low_idxs = idx_sorted[:3]
high_idxs = idx_sorted[-3:][::-1]

print("Lowest leakage_ratio (mid):")
for i in low_idxs:
    print(i, df.loc[i, "image_id"], "ratio_mid=", float(rank_ratio[i]), "outside_mid=", float(outside_mid[i]), "inside_mid=", float(inside_mid[i]))

print("\nHighest leakage_ratio (mid):")
for i in high_idxs:
    print(i, df.loc[i, "image_id"], "ratio_mid=", float(rank_ratio[i]), "outside_mid=", float(outside_mid[i]), "inside_mid=", float(inside_mid[i]))


In [ ]:
def show_token_change_mask(row_idx: int, level: str = "mid"):
    row = df.loc[row_idx]
    H, W = row["token_grid_hw"]
    j0, i0, j1, i1 = row["patch_bbox_tok"]

    zc = np.array(row["indices_clean"], dtype=np.int32).reshape(H, W)
    zn = np.array(row[f"indices_{level}"], dtype=np.int32).reshape(H, W)
    delta = (zn != zc).astype(np.float32)

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    im = ax.imshow(delta, vmin=0, vmax=1)  # 0=no change, 1=change
    ax.set_title(f"Token ID change mask ({level})")
    ax.axis("off")

    # patch bbox overlay in token coords
    ax.add_patch(plt.Rectangle((j0-0.5, i0-0.5), (j1-j0), (i1-i0),
                               fill=False, linewidth=2))
    plt.tight_layout()
    plt.show()

# Example: show one low and one high
show_token_change_mask(int(low_idxs[0]), level="mid")
show_token_change_mask(int(high_idxs[0]), level="mid")


In [ ]:
def idx_from_image_id(image_id: str) -> int:
    hits = df.index[df["image_id"] == image_id].to_list()
    assert len(hits) == 1, f"Expected exactly 1 match for {image_id}, got {len(hits)}"
    return int(hits[0])

low_id  = "n04086273/ILSVRC2012_val_00032525"
high_id = "n02233338/ILSVRC2012_val_00024082"

low_idx  = idx_from_image_id(low_id)
high_idx = idx_from_image_id(high_id)

print("low_idx:", low_idx, "high_idx:", high_idx)


In [ ]:
def show_token_masks_3(row_idx: int):
    row = df.loc[row_idx]
    H, W = row["token_grid_hw"]
    j0, i0, j1, i1 = row["patch_bbox_tok"]

    zc = np.array(row["indices_clean"], dtype=np.int32).reshape(H, W)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, lvl in zip(axes, ["low", "mid", "high"]):
        zn = np.array(row[f"indices_{lvl}"], dtype=np.int32).reshape(H, W)
        delta = (zn != zc).astype(np.float32)

        ax.imshow(delta, vmin=0, vmax=1)
        ax.add_patch(plt.Rectangle((j0-0.5, i0-0.5), (j1-j0), (i1-i0),
                                   fill=False, linewidth=2))
        ax.set_title(f"Token ID change mask ({lvl})")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

print("LOW leakage sample masks:")
show_token_masks_3(low_idx)

print("HIGH leakage sample masks:")
show_token_masks_3(high_idx)


In [ ]:
def show_full_2x4_for_idx(row_idx: int, overlay=True):
    row = df.loc[row_idx].to_dict()
    image_id = row["image_id"]
    files = find_full_set(image_id)

    print("\nimage_id:", image_id)
    print("found:", {k: files[k].name for k in sorted(files.keys())})

    needed = list(range(8))
    assert all(k in files for k in needed), "Not all 0..7 files found."

    titles_top = ["original", "input low", "input mid", "input high"]
    titles_bot = ["recon clean", "recon low", "recon mid", "recon high"]

    top_paths = [files[0], files[2], files[4], files[6]]
    bot_paths = [files[1], files[3], files[5], files[7]]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for c, p in enumerate(top_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[0, c]
        ax.imshow(img)
        ax.set_title(titles_top[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_boxes(ax, img.width, img.height,
                       row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])

    for c, p in enumerate(bot_paths):
        img = Image.open(p).convert("RGB")
        ax = axes[1, c]
        ax.imshow(img)
        ax.set_title(titles_bot[c], fontsize=10)
        ax.axis("off")
        if overlay:
            draw_boxes(ax, img.width, img.height,
                       row["patch_bbox_px"], row["patch_bbox_tok"], row["token_grid_hw"])

    plt.tight_layout()
    plt.show()

print("LOW leakage sample recon grid:")
show_full_2x4_for_idx(low_idx, overlay=True)

print("HIGH leakage sample recon grid:")
show_full_2x4_for_idx(high_idx, overlay=True)


### Results: Local vs Global Token-ID Churn

We visualise token-ID change masks for a low-leakage and a high-leakage sample.
In each mask, a token is marked as “changed” if $z_{\sigma}(i,j) \neq z(i,j)$ (yellow), otherwise unchanged (dark).
The black box denotes the token-aligned patch region $\mathcal{P}$.

### Low leakage example (stable outside the patch)
For the low-leakage sample, token changes are concentrated inside $\mathcal{P}$ across noise levels, with only sparse changes outside.
This matches the quantitative summary (e.g., at $\sigma=0.2$, $\text{outside\_change}\approx 0.09$ and $\text{leakage\_ratio}\approx 0.11$), indicating largely localised encoder response.

### High leakage example (global token reassignment)
For the high-leakage sample, token changes extend far beyond $\mathcal{P}$ and become nearly global as $\sigma$ increases.
This aligns with extreme leakage statistics (e.g., at $\sigma=0.2$, $\text{outside\_change}\approx 0.78$ and $\text{leakage\_ratio}\approx 0.79$), demonstrating that patch-localised pixel noise can trigger widespread token-ID reassignment.

### Visual reconstructions vs token stability
Interestingly, the reconstruction grids suggest that visible corruption often remains concentrated near the patch even when token-ID changes are widespread.
Therefore, H1 primarily measures **encoder token stability / locality**, which can diverge from perceptual stability in the decoded image.


## Summary (H1: Encoder Locality under Patch-Localised Pixel Noise)

We evaluate how a pixel-space noise patch affects **discrete encoder token IDs**. For each image, we compare the clean token grid
$z \in \{0,\dots,K-1\}^{H\times W}$ with the noisy token grid $z_{\sigma}$ and mark a token as changed if
$$
\Delta_{\sigma}(i,j) = \mathbf{1}\!\left[z_{\sigma}(i,j)\neq z(i,j)\right].
$$
The patch is defined in token space by a bounding box $B=(j_0,i_0,j_1,i_1)$ with inside set $\mathcal{P}$ and complement $\mathcal{P}^c$.

### Core metrics (dataset-level)
We report:
$$
\text{inside\_change}(\sigma)=\frac{1}{|\mathcal{P}|}\sum_{(i,j)\in\mathcal{P}}\Delta_{\sigma}(i,j),
\qquad
\text{outside\_change}(\sigma)=\frac{1}{|\mathcal{P}^c|}\sum_{(i,j)\in\mathcal{P}^c}\Delta_{\sigma}(i,j),
$$
and the normalised leakage ratio
$$
\text{leakage\_ratio}(\sigma)=\frac{\text{outside\_change}(\sigma)}{\text{inside\_change}(\sigma)+\varepsilon}.
$$

Across the dataset, inside changes are high and quickly saturate with noise:
- at $\sigma=0.1$, median inside-change $\approx 0.77$
- at $\sigma=0.5$, median inside-change $\approx 0.99$

Outside changes are **substantial** and increase with $\sigma$:
- at $\sigma=0.1$, median outside-change $\approx 0.27$
- at $\sigma=0.5$, median outside-change $\approx 0.49$

Leakage ratios are correspondingly large:
- at $\sigma=0.1$, median leakage-ratio $\approx 0.36$
- at $\sigma=0.5$, median leakage-ratio $\approx 0.50$

These results indicate that patch-localised pixel noise produces widespread token-ID changes beyond the patch region.

### Spatial leakage profile (distance-to-patch curve)
To distinguish boundary spillover from long-range effects, we compute Manhattan distance $d(i,j)$ (in token units) to the patch and estimate
$$
p_{\sigma}(d)
=
\mathbb{E}\left[
\frac{1}{|\{(i,j): d(i,j)=d\}|}
\sum_{(i,j):d(i,j)=d}\Delta_{\sigma}(i,j)
\right],
\qquad d=0,1,2,\dots
$$

We observe a sharp decay close to the patch ($d=0\to 1\to 2\to 3$), followed by a **non-zero plateau** at larger distances:
- plateau mean over $d\in[8,12]$ is $\approx 0.16$ at $\sigma=0.1$, $\approx 0.22$ at $\sigma=0.2$, and $\approx 0.32$ at $\sigma=0.5$.

Thus, tokens far from the patch retain a non-trivial change probability that grows with $\sigma$, evidencing strong non-local sensitivity.

### Boundary vs far-field decomposition
We decompose changes by distance bands:
- boundary ring: $d=1$
- near-field: $1 \le d \le 2$
- far-field: $d \ge 6$

Far-field leakage remains significant:
- at $\sigma=0.1$, far-field ( $d\ge 6$ ) mean $\approx 0.164$ (median $\approx 0.154$)
- at $\sigma=0.2$, far-field mean $\approx 0.233$ (median $\approx 0.229$)
- at $\sigma=0.5$, far-field mean $\approx 0.326$ (median $\approx 0.314$)

This confirms that leakage is not confined to the boundary; a substantial fraction persists well away from the patch.

### Qualitative evidence and perceptual note
Token-ID change masks show two regimes:
- **low leakage** samples: changes concentrate inside the patch with sparse outside flips,
- **high leakage** samples: changes become near-global even at moderate noise.

Comparing input/reconstruction grids suggests that widespread token-ID churn can occur even when visible reconstruction corruption is mostly localised near the patch. Therefore, H1 primarily diagnoses **encoder token stability/locality**, which may not map one-to-one to perceptual degradation in reconstructions.

### Conclusion
Adding noise to just one local patch of the image predictably scrambles the encoder’s tokens inside that patch, but it also causes a surprisingly large ripple through the rest of the image: even far away from the patch, a non-trivial fraction of token IDs flip, and this “global churn” gets stronger as the noise increases. The distance-to-patch analysis shows a quick drop right outside the patch, then a clear non-zero plateau rather than fading to zero, meaning the effect is not just a boundary spill. Some images behave fairly locally, but others go almost fully global, with most tokens outside the patch changing as well. Visually, reconstructions often look mostly corrupted only inside the patch even when the token IDs change widely, so the main conclusion is about encoder token stability and locality rather than purely perceptual image quality.



Interpretation: the edit operation is behaving exactly as designed. We are editing almost exclusively the intended patch, with tiny discrete deviations explainable by bbox rounding / clipping near image borders.